<a href="https://colab.research.google.com/github/Komal5578/DWM/blob/main/NAIVE_BAYES_WITHOUT_LIB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Features: Pregnancies, Glucose, BloodPressure, SkinThickness,
# Insulin, BMI, DiabetesPedigreeFunction, Age
# Target: Outcome (0 = No Diabetes, 1 = Diabetes)
data = [
    [1, 85, 66, 29, 0, 26.6, 0.351, 31, 0],
    [8, 183, 64, 0, 0, 23.3, 0.672, 32, 1],
    [1, 89, 66, 23, 94, 28.1, 0.167, 21, 0],
    [0, 137, 40, 35, 168, 43.1, 2.288, 33, 1],
    [5, 116, 74, 0, 0, 25.6, 0.201, 30, 0],
    [3, 78, 50, 32, 88, 31.0, 0.248, 26, 1],
    [10, 115, 0, 0, 0, 35.3, 0.134, 29, 0],
    [2, 197, 70, 45, 543, 30.5, 0.158, 53, 1],
    [4, 110, 92, 0, 0, 37.6, 0.191, 30, 0],
    [7, 196, 90, 0, 0, 39.8, 0.451, 41, 1]
]

In [2]:
# First 7 rows for training, last 3 for testing (about 70:30)
train_data = data[:7]
test_data = data[7:]

print("Training records:", len(train_data))
print("Testing records:", len(test_data))

Training records: 7
Testing records: 3


In [3]:
# Group the rows by class label: {0: [rows], 1: [rows]}
# The label (last column) is dropped from each row
def separate_by_class(data):
    separated = {}
    for row in data:
        label = row[-1]
        if label not in separated:
            separated[label] = []
        separated[label].append(row[:-1])
    return separated

In [4]:
# Mean = sum / count
def mean(values):
    return sum(values) / len(values)

# Variance = average of squared distances from the mean
def variance(values):
    m = mean(values)
    return sum((x - m) ** 2 for x in values) / len(values)

In [5]:
# Normal distribution formula: gives P(x | class) for a numeric feature
def gaussian_probability(x, m, v):
    if v == 0:                      # avoid division by zero
        return 1 if x == m else 1e-10
    exponent = -((x - m) ** 2) / (2 * v)
    return (1 / ((2 * 3.14159 * v) ** 0.5)) * (2.71828 ** exponent)

In [6]:
# For each class, store (mean, variance) of every feature
def train(data):
    separated = separate_by_class(data)
    model = {}
    for label, rows in separated.items():
        model[label] = []
        for column in zip(*rows):   # zip(*rows) gives one column at a time
            m = mean(column)
            v = variance(column)
            model[label].append((m, v))
    return model

In [7]:
# Score = P(class) x P(f1|class) x P(f2|class) x ...
# The class with the highest score is the prediction
def predict(model, input_data, total):
    probabilities = {}
    for label, features in model.items():
        count = len(model_data[label])
        probability = count / total              # prior probability
        for i in range(len(features)):
            m, v = features[i]
            p = gaussian_probability(input_data[i], m, v)
            probability *= p                     # multiply likelihoods
        probabilities[label] = probability
    return max(probabilities, key=probabilities.get)

In [8]:
# model_data is used by predict() to count rows per class
model_data = separate_by_class(train_data)
model = train(train_data)
print("MODEL TRAINED SUCCESSFULLY!")

print("\nMEAN AND VARIANCE")
for label, features in model.items():
    print("\nClass:", label)
    for i, (m, v) in enumerate(features):
        print("Feature", i + 1, "Mean:", round(m, 2), "Variance:", round(v, 2))

MODEL TRAINED SUCCESSFULLY!

MEAN AND VARIANCE

Class: 0
Feature 1 Mean: 4.25 Variance: 13.69
Feature 2 Mean: 101.25 Variance: 205.19
Feature 3 Mean: 51.5 Variance: 894.75
Feature 4 Mean: 13.0 Variance: 173.5
Feature 5 Mean: 23.5 Variance: 1656.75
Feature 6 Mean: 28.9 Variance: 14.44
Feature 7 Mean: 0.21 Variance: 0.01
Feature 8 Mean: 27.75 Variance: 15.69

Class: 1
Feature 1 Mean: 3.67 Variance: 10.89
Feature 2 Mean: 132.67 Variance: 1846.89
Feature 3 Mean: 51.33 Variance: 96.89
Feature 4 Mean: 22.33 Variance: 250.89
Feature 5 Mean: 85.33 Variance: 4707.56
Feature 6 Mean: 32.47 Variance: 66.42
Feature 7 Mean: 1.07 Variance: 0.77
Feature 8 Mean: 30.33 Variance: 9.56


In [9]:
# Predict every test row and compare with the actual label
y_actual = []
y_predicted = []

for row in test_data:
    actual = row[-1]
    prediction = predict(model, row[:-1], len(train_data))
    y_actual.append(actual)
    y_predicted.append(prediction)
    print("Actual:", actual, "Predicted:", prediction)

Actual: 1 Predicted: 1
Actual: 0 Predicted: 0
Actual: 1 Predicted: 1


In [10]:
# Accuracy = correct predictions / total predictions
correct = 0
for i in range(len(y_actual)):
    if y_actual[i] == y_predicted[i]:
        correct += 1

accuracy = correct / len(y_actual)
print("ACCURACY:", round(accuracy * 100, 2), "%")

ACCURACY: 100.0 %


In [11]:
# TP = predicted 1, actually 1    TN = predicted 0, actually 0
# FP = predicted 1, actually 0    FN = predicted 0, actually 1
TP = TN = FP = FN = 0

for i in range(len(y_actual)):
    if y_actual[i] == 1 and y_predicted[i] == 1:
        TP += 1
    elif y_actual[i] == 0 and y_predicted[i] == 0:
        TN += 1
    elif y_actual[i] == 0 and y_predicted[i] == 1:
        FP += 1
    else:
        FN += 1

print("CONFUSION MATRIX")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

CONFUSION MATRIX
TP: 2
TN: 1
FP: 0
FN: 0


In [12]:
# Order: Pregnancies, Glucose, BP, Skin, Insulin, BMI, DPF, Age
new_patient = [2, 120, 70, 25, 80, 28.5, 0.5, 30]

result = predict(model, new_patient, len(train_data))
print("NEW PATIENT:", new_patient)

if result == 1:
    print("PREDICTION: DIABETES")
else:
    print("PREDICTION: NO DIABETES")

NEW PATIENT: [2, 120, 70, 25, 80, 28.5, 0.5, 30]
PREDICTION: DIABETES
